# Efficient VLA on Kaggle: SmolVLA + LIBERO-Spatial

Clean copy of the notebook used for every run in this repository.
Settings: accelerator **GPU T4 x2**, Internet **on**, persistence off.
Optional secret `GH_TOKEN` (fine-grained, contents read/write on the repo) lets
`scripts/push_results.py` push partial results to GitHub every 20 minutes; without it, results stay
in `/kaggle/working/results`.

## 1. Code and pinned environment (about 6 minutes)

In [ ]:
!git clone -q https://github.com/Ngoc-LM/efficient-vla
%cd efficient-vla
!git log --oneline -1
!bash scripts/kaggle_setup.sh 2>&1 | tail -n 8

## 2. Profiling (isolated calls, batch 1)
Component breakdown of one re-plan, then re-plan latency over precision x Euler steps x eager/CUDA graph.

In [ ]:
!PYTHONPATH=src python -m evla.profile --out /kaggle/working/results/profile_t4.json 2>&1 | tail -n 20
!PYTHONPATH=src python -m evla.bench_graph --out /kaggle/working/results/graph_t4.json 2>&1 | tail -n 20

## 3. Closed-loop runs
Each line of a queue file is `<run name> <run_eval options>`; runs execute one after another as four
processes on the two GPUs and are detached from the cell, so the notebook stays usable.
Re-running a queue resumes it (finished episodes are skipped).

In [ ]:
!bash scripts/bg.sh /kaggle/working/results/queue.log bash scripts/run_queue.sh scripts/queues/queue_spatial_v6.txt

Monitoring. Kaggle stops an interactive session after about 40 minutes without interaction, so keep
the editor active (this loop prints progress every 10 minutes; interrupt it with the stop button,
which does not stop the detached runs).

In [ ]:
!for n in $(seq 1 70); do echo "$(date +%H:%M) $(tail -n1 /kaggle/working/results/queue.log | cut -c1-60) | $(ls -t /kaggle/working/results/raw/*_s*.log 2>/dev/null | head -4 | xargs -r -n1 tail -n1 | grep -o '[0-9]*/[0-9]* SR=[0-9.]*' | tr '\n' ' ')"; sleep 600; done

## 4. Paired analysis (CPU is enough; also runs locally on `results/t4/raw`)

In [ ]:
!PYTHONPATH=src python -m evla.analyze --root /kaggle/working/results/raw